In [1]:
from torchvision.datasets import GTSRB
from torchvision import transforms 
import torch
import torch.nn as nn
from torch.optim import Adam
import  torchvision.transforms.v2 as transforms
import torchvision.transforms.functional as F 
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

In [2]:
from pathlib import Path

print(Path.cwd())
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

DATA_DIR.mkdir(exist_ok=True)


C:\Users\user\deep-learning-traffic-signs\notebooks


In [3]:
basic_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToImage(),                      
    transforms.ToDtype(torch.float32, scale=True),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=.2 , contrast=0.5),
    transforms.RandomResizedCrop((64,64),scale=(0.9,1),ratio=(1,1)),
])
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToImage(),                        
    transforms.ToDtype(torch.float32, scale=True)
])
train_dataset = GTSRB(
    root=str(DATA_DIR),
    split="train",
    transform=basic_transform,
    download=True
)

test_dataset = GTSRB(
    root=str(DATA_DIR),
    split="test",
    transform=test_transform,
    download=True
)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
 
)
train_N=len(train_loader.dataset)
test_N=len(test_loader.dataset)

In [4]:
class MyConvBlock( nn.Module ):
    def __init__(self,in_ch,out_ch,dropout_p):
        kernel_size=3
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,kernel_size,stride=1,padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.MaxPool2d(2,stride=2)
        )
    def forward(self,x):
        return self.model(x)
flattened_img_size=128*8*8
N_CLASSES=43
IMG_CHS=3
IMG_WIDH=64
IMG_LENGHT=64
base_model= nn.Sequential(
    MyConvBlock(IMG_CHS,32,0), #(32,32,32)
    MyConvBlock(32,64,0.2),#(64,16,16)
    MyConvBlock(64,128,0),#(128,8,8)
    nn.Flatten(),
    nn.Linear(flattened_img_size,1024),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(1024,N_CLASSES)
)
loss_function=nn.CrossEntropyLoss()
optimizer=Adam(base_model.parameters())
torch._dynamo.config.suppress_errors = True
device=torch.device("cpu")
model=torch.compile(base_model.to(device))
model

OptimizedModule(
  (_orig_mod): Sequential(
    (0): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0, inplace=False)
        (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0.2, inplace=False)
        (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(128, eps=1e

In [5]:
def get_batch_accuracy(output,y,N):
    pred=output.argmax(dim=1,keepdim=True)
    correct=pred.eq(y.view_as(pred)).sum().item()
    return correct/N


In [6]:
def train():
    loss=0
    accuracy=0
    model.train()
    for x,y in train_loader:
        output=model(x)
        optimizer.zero_grad()
        batch_loss=loss_function(output,y)
        batch_loss.backward()
        optimizer.step()
        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output,y,train_N)
    print('Train-Loss:{:.4f} Accuracy {:.4f}'.format(loss,accuracy))
    

In [7]:
def validate():
    loss=0
    accuracy=0
    model.eval()
    with torch.no_grad():
        for x,y in test_loader:
            output=model(x)
            loss+=loss_function(output,y).item()
            accuracy += get_batch_accuracy(output,y,test_N)
        print('Valid-Loss:{:.4f} Accuracy {:.4f}'.format(loss,accuracy))

In [8]:
epochs=15
for epoch in range (epochs):
    print ('epoch:{}'.format(epoch))   
    train()
    validate()

epoch:0


W0923 18:32:19.360000 19772 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] WON'T CONVERT inner C:\Users\user\deep-learning-traffic-signs\.venv\Lib\site-packages\torch\_dynamo\external_utils.py line 67 
W0923 18:32:19.360000 19772 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] due to: 
W0923 18:32:19.360000 19772 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] Traceback (most recent call last):
W0923 18:32:19.360000 19772 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]   File "C:\Users\user\deep-learning-traffic-signs\.venv\Lib\site-packages\torch\_dynamo\convert_frame.py", line 2333, in __call__
W0923 18:32:19.360000 19772 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]     result = self._inner_convert(
W0923 18:32:19.360000 19772 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]         frame, cache_entry, hooks, frame_state, skip=skip + 1
W0923 18:32:19.360000 19772 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]     )
W0923 18:32:19.3

Train-Loss:1331.4251 Accuracy 0.5363
Valid-Loss:181.1278 Accuracy 0.8629
epoch:1
Train-Loss:269.3250 Accuracy 0.8956
Valid-Loss:99.8941 Accuracy 0.9211
epoch:2
Train-Loss:152.8728 Accuracy 0.9426
Valid-Loss:70.3939 Accuracy 0.9496
epoch:3
Train-Loss:106.2748 Accuracy 0.9607
Valid-Loss:59.7756 Accuracy 0.9548
epoch:4
Train-Loss:77.5923 Accuracy 0.9711
Valid-Loss:56.0743 Accuracy 0.9641
epoch:5
Train-Loss:66.3519 Accuracy 0.9759
Valid-Loss:52.8661 Accuracy 0.9623
epoch:6
Train-Loss:57.3278 Accuracy 0.9779
Valid-Loss:49.1491 Accuracy 0.9652
epoch:7
Train-Loss:52.7572 Accuracy 0.9798
Valid-Loss:44.8521 Accuracy 0.9691
epoch:8
Train-Loss:40.3042 Accuracy 0.9848
Valid-Loss:44.3252 Accuracy 0.9692
epoch:9
Train-Loss:34.7222 Accuracy 0.9881
Valid-Loss:40.3147 Accuracy 0.9728
epoch:10
Train-Loss:31.2299 Accuracy 0.9890
Valid-Loss:40.9814 Accuracy 0.9682
epoch:11
Train-Loss:30.0909 Accuracy 0.9891
Valid-Loss:44.2626 Accuracy 0.9709
epoch:12
Train-Loss:28.2427 Accuracy 0.9898
Valid-Loss:42.7461 A

In [9]:
torch.save(base_model,'model.pth')